In [126]:
# preprocessing (extracting the text and removing problematic characters)
# poetry run spacy download en_core_web_sm
# https://github.com/keredson/wordninja
from pypdf import PdfReader, mult
from nlp_text_clean.cleaner import TextCleaner
import wordninja
import re
import pymupdf4llm
import pathlib
import base64
import pymupdf
import io

path = pathlib.Path('../text/eng-kjv_MAT.pdf')
cleaner = TextCleaner(
    remove_punctuation=True,
    remove_numbers=True,
    remove_special_chars=True,
    use_stemming=False,
    use_lemmatization=True,
    language="english",
    custom_stopwords={
        "unto", "shall", "ye", "thou", "thy", "thee", "hath",
        "doth", "didst", "shalt", "art", "hast", "saith",
        "spake", "wilt", "say", "said", "david",
    },
    preserve_num_token=False,
    remove_html_tags=True,
    remove_urls=True
)

with pymupdf.open(path) as doc:
    book_markdown_text = pymupdf4llm.to_markdown(doc)

    with open('../temp/test.md', 'w') as file:
        file.write(book_markdown_text)

In [127]:
# tokenizing (making the text divided into even smaller chunks)
# https://www.nltk.org/api/nltk.tokenize.punkt.html
# https://www.nltk.org/
import nltk
from nltk.tokenize import sent_tokenize

# downloading pre treined punkt model
nltk.download('punkt')

ROMAN_ONLY = re.compile(
    r"^m{0,4}(?:cm|cd|d?c{0,3})(?:xc|xl|l?x{0,3})(?:ix|iv|v?i{0,3})$",
    re.IGNORECASE,
)

# tokinizing and cleaning
with open('../temp/test.md', 'r') as file:
    docs = sent_tokenize(file.read())
    docs = [cleaner.clean_text(doc) for doc in docs]

    # deconstruct and reconstruct without roman algs
    for i in range(len(docs)):
        words = list(filter(lambda x: not re.match(ROMAN_ONLY, x), docs[i].split(' ')))
        docs[i] = ' '.join(words)

    # replace remaining header (matthew matthew)
    for i in range(len(docs)):
        docs[i] = docs[i].replace('matthew matthew', '')

# vetorizing
# https://www.geeksforgeeks.org/nlp/vectorization-techniques-in-nlp/
# https://www.geeksforgeeks.org/nlp/5-simple-ways-to-tokenize-text-in-python/
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)


[nltk_data] Downloading package punkt to /home/arthur/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [128]:
print(X.shape)                    # (n_verses, n_vocab)
print(vectorizer.get_feature_names_out()[:15])
print(vectorizer.idf_[:15])

print("Matrix shape:", X.shape)
print("Non-zero entries:", X.nnz)
print("Sparsity: {:.2%}".format(1 - X.nnz / (X.shape[0] * X.shape[1])))
print("Vocab size:", len(vectorizer.vocabulary_))

# What are the highest-IDF (rarest, most "informative") tokens?
idf = vectorizer.idf_
vocab = vectorizer.get_feature_names_out()
rare = vocab[np.argsort(idf)[-20:]]
print("Rarest tokens:", rare[::-1])

# What are the lowest-IDF (most common, least informative) tokens?
common = vocab[np.argsort(idf)[:20]]
print("Most common tokens:", common)

(1046, 1668)
['abase' 'abel' 'abia' 'abide' 'ability' 'abiud' 'able' 'abode'
 'abomination' 'abound' 'abraham' 'abroad' 'abundance' 'accord' 'account']
[7.26053703 7.26053703 7.26053703 7.26053703 7.26053703 7.26053703
 5.75645963 7.26053703 7.26053703 7.26053703 6.00777406 6.16192474
 6.56738985 6.16192474 6.85507192]
Matrix shape: (1046, 1668)
Non-zero entries: 8743
Sparsity: 99.50%
Vocab size: 1668
Rarest tokens: ['zacharias' 'abase' 'zorobabel' 'abode' 'abiud' 'ability' 'abide' 'abia'
 'zara' 'herb' 'henceforward' 'hen' 'help' 'heir' 'hedge' 'heavens'
 'alway' 'alphaeus' 'withdraw' 'winter']
Most common tokens: ['come' 'jesus' 'man' 'go' 'say' 'take' 'see' 'son' 'one' 'disciple'
 'lord' 'heaven' 'give' 'behold' 'day' 'father' 'thing' 'kingdom' 'answer'
 'hear']


In [129]:
from sklearn.metrics.pairwise import cosine_similarity

# Similarity between ALL pairs (n_verses × n_verses)
sim = cosine_similarity(X)

# Most similar verse to verse 0 (excluding itself)
def most_similar(i, sim, topn=5):
    scores = sim[i].copy()
    scores[i] = -1   # exclude self
    top = np.argsort(scores)[-topn:][::-1]
    return [(j, scores[j]) for j in top]

verse_texts = [page for page in docs if page]

for j, s in most_similar(0, sim):
    print(f"{s:.3f}  {verse_texts[j][:80]}")

def search(query, vectorizer, X, verse_texts, topn=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, X).ravel()
    top = np.argsort(scores)[-topn:][::-1]
    for j in top:
        print(f"{scores[j]:.3f}  {verse_texts[j][:90]}")

search("angel appear joseph dream", vectorizer, X, verse_texts)
# → should surface the annunciation + flight-to-Egypt verses

0.377  bind heavy burden grievous bear lie man shoulder move one finger
0.243  generation abraham fourteen generation carry away babylon fourteen generation ca
0.215  verily standing taste death till see son man come kingdom
0.213  lord therefore vineyard cometh husbandman
0.207  jesus answer bless simon bar jona flesh blood reveal father heaven
0.534  think thing behold angel lord appear dream say joseph son fear take mary wife conceive hol
0.323  depart behold angel lord appeareth joseph dream say arise take young child mother flee egy
0.306  herod dead behold angel lord appeareth dream joseph egypt say arise take young child mothe
0.252  joseph raise sleep angel lord bid take wife know till bring forth firstborn son call name 
0.223  answer peter jesus lord good we let we make three tabernacle one one moses one elia
